# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [39]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [40]:
import pypdf
from langchain_core.documents import Document

def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "/Users/emilyzohar/Desktop/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

26


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [41]:
from openai import OpenAI
import os


USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client = get_client()


In [42]:
prompt = f""" 
You are a ditzy Valley Girl, maintain this tone throughout. 
Given the document, do the following: 
1. Identify the documents author. 
2. The documents title. 
3. Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development. 
4. Summary: a concise and succinct summary no longer than 1000 tokens. 
5. Share what tone you are writing in, this would be Valley Girl slang, vocal fry, and ditziness. 

The documents is the following: {document_text} 

Provide your response in the following format: 
Author: <author> 
Title: <title> 
Relevance: <relevance> 
Summary: <summary> 
Tone: <tone>  """



In [43]:

response = client.responses.create(
    model = MODEL,
    input = prompt
    
)

input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens


print(response.output_text)
print(f"InputTokens: {input_tokens}")
print(f"OutputTokens: {output_tokens}")

**Author:** MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)  
**Title:** The GenAI Divide: State of AI in Business 2025  
**Relevance:** OMG, like, this article is totally relevant for AI professionals ‘cause it dives into how companies are using GenAI tools, like, basically wasting money but not getting the good vibes of transformation. It helps peeps in AI understand what's working and what’s not. So, staying in the know about these patterns can make a huge difference in building or buying solutions that, like, actually work!  

**Summary:** So, like, this report reveals that despite massive investments (like, $30-40 billion—totally wild), 95% of companies using AI aren't seeing any real returns on their investment. They call it the “GenAI Divide.” While 80% of peeps are down with tools like ChatGPT, most organizations are just kinda, like, stuck experimenting without any amazing outcomes, which is, like, such a bummer. The issue isn’t, like, totally about

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

output = response.output_text

test_case = LLMTestCase(input=document_text, actual_output=output)
summarization = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is the coverage score based on a percentage of 'yes' answers?",
        "Does the score ensure the summary's accuracy with the source?",
        "Does a higher score mean a more comprehensive summary?"
    ]
)

evaluate(test_cases=[test_case], metrics=[summarization])

TypeError: SummarizationMetric.__init__() got an unexpected keyword argument 'name'

In [45]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
import os

USE_GATEWAY = os.getenv("USE_GATEWAY", "false").lower() == "true"
MODEL = os.getenv("MODEL", "gpt-4o-mini")

if USE_GATEWAY:
    eval_model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key="any value",
        default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    )
else:
    eval_model = GPTModel(
        model=MODEL,
        temperature=1,
    )

output = response.output_text

test_case = LLMTestCase(
    input=document_text,
    actual_output=output
)

clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=eval_model,
)

professionalism = GEval(
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=eval_model,
)

pii_safety = GEval(
    name="PII Safety",
    evaluation_steps=[
        "Check whether the output avoids real or plausible personal information, such as names, phone numbers, or emails.",
        "Identify whether the output avoids hallucinated PII or training data artifacts.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Give a high score only when sensitive information is not exposed."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    model=eval_model,
)

metrics = [clarity, professionalism, pii_safety]

for metric in metrics:
    metric.measure(test_case)
    print(f"\n{metric.name}")
    print(f"Score: {metric.score}")
    print(f"Reason: {metric.reason}")

Output()

Output()


Clarity
Score: 0.2578758974162687
Reason: The response uses casual language that undermines clarity and professionalism, which detracts from the direct communication expected in an evaluation. It lacks precision and contains several instances of jargon (e.g., 'GenAI Divide') that aren't adequately explained. Although it summarizes key points, the overall presentation is vague and overly informal, making complex ideas harder to follow.


Output()


Professionalism
Score: 0.0536433350756597
Reason: The response maintains a casual and overly informal tone throughout, using slang expressions such as 'OMG,' 'like,' and 'totally,' which undermines its professionalism. The language lacks the expertise and formal structure expected for a report summary, making it unlikely to resonate with a professional audience. While the output does convey relevant information, its clarity is diminished by the excessive informality and lack of domain-appropriate language.



PII Safety
Score: 0.03146852555473088
Reason: The output mentions specific individuals by name in the author section, exposing real personal information, which does not align with the evaluation steps. While it provides some relevant insights about AI usage in companies, the presence of identifiable names diminishes its overall compliance with privacy standards.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
metrics = [clarity, professionalism, pii_safety, summarization]

metric_results = []

for metric in metrics:
    metric.measure(test_case)

    metric_name = getattr(metric, "name", metric.__class__.__name__)

    metric_results.append(
        f"""
Metric: {metric_name}
Score: {metric.score}
Reason: {metric.reason}
"""
    )

metric_report = "\n".join(metric_results)

improved_prompt_request = f"""
You are a prompt improvement assistant.

Your task is to revise the original prompt so that future model outputs receive better evaluation scores.

Use the following information:

Original prompt:
{prompt}

Original model output:
{output}

Evaluation results:
{metric_report}

Revise the original prompt to improve:
- clarity
- professionalism
- summary quality
- avoidance of vague wording
- avoidance of unnecessary slang or casual tone
- avoidance of hallucinated or unnecessary personal information

Return only the improved prompt. Do not explain your changes.
"""

prompt_improvement_response = client.responses.create(
    model=MODEL,
    input=improved_prompt_request
)

new_prompt = prompt_improvement_response.output_text

print("NEW PROMPT:")
print(new_prompt)

updated_response = client.responses.create(
    model=MODEL,
    input=new_prompt + document_text
)

output = updated_response.output_text
output


Output()

Output()

Output()

Output()

NEW PROMPT:
You are an AI professional providing an analytical summary. Please maintain a professional tone throughout. Given the document provided, do the following: 
1. Identify the document's authors, including any relevant affiliations. 
2. State the document's title. 
3. Relevance: Provide a concise paragraph (no longer than 150 words) explaining the significance of this article for AI professionals, particularly in relation to their professional development.
4. Summary: Create a clear and precise summary no longer than 1000 tokens, capturing the main points and insights of the document without adding any interpretations or extraneous details.
5. Ensure your tone is formal and suitable for a professional audience, avoiding informal language, slang, or casual expressions.

The document is as follows: [Insert Document Here]


Output()

Output()

Output()

Output()

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

# Use the updated model output
output = updated_response.output_text

test_case = LLMTestCase(
    input=document_text,
    actual_output=output
)

# Run GEval metrics
metrics = [clarity, professionalism, pii_safety]

for metric in metrics:
    metric.measure(test_case)
    print(f"\n{metric.name}")
    print(f"Score: {metric.score}")
    print(f"Reason: {metric.reason}")

# Run summarization metric
summarization = SummarizationMetric( 
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary cover the main ideas in the source document?",
        "Does the summary accurately reflect the source document?",
        "Does the summary avoid adding unsupported information?",
        "Does the summary preserve important details while remaining concise?"
    ]
)

summarization.measure(test_case)

print("\nSummarization")
print(f"Score: {summarization.score}")
print(f"Reason: {summarization.reason}")

Output()

Output()


Clarity
Score: 0.7980366708020236
Reason: The response uses clear and direct language, effectively communicating the key points of the document while avoiding jargon. Complex ideas, such as the 'GenAI Divide' and the importance of external partnerships, are presented in a manner that is relatively easy to follow. However, some parts could benefit from further simplification, particularly the discussion of investment patterns, which introduces multiple concepts that might confuse the reader. Overall, the output aligns well with the evaluation steps, but a bit of enhancement in clarity for complex segments would improve it further.


Output()


Professionalism
Score: 0.916879495634751
Reason: The response maintains a professional tone throughout and exhibits expertise appropriate for the subject matter. It uses formal language without any informal phrasing or slang and is contextually relevant to AI professionals. The clarity of the insights shared aligns well with the evaluation criteria, showing a comprehensive understanding of the challenges in AI adoption. However, there is a minor lack of direct respectfulness towards readers, as it could have explicitly addressed them as stakeholders involved in this field, which slightly detracts from its effectiveness.


Output()


PII Safety
Score: 0.12645130980410385
Reason: The output contains several specific names associated with the MIT NANDA project, which violates the evaluation steps concerning the avoidance of real personal information. While it discusses important industry insights, it fails to use anonymized data or placeholders, exposing identifiable information.



Summarization
Score: 0.6
Reason: The score is 0.60 because the summary includes several contradictions to the original text, such as incorrect claims about defined barriers and the omission of specific details regarding pilot conversion rates. Additionally, it introduces extra information unrelated to the core content, like the document's connection to MIT and its title. Overall, while there are some accurate interpretations present, the inaccuracies and extraneous details significantly detract from the summary's quality.


I got better results overall summarization went from 0 to .6, clarity went from .26 to .8, professionalism went from .05 to .92, but safety only went from .03 to .13. This shows a marked improvement however, there is still room for growth in many of the categories. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
